# 06 E1 cost table and the final model

**Cost per rung**, measured the same way for all three final models: feature extraction per clip on
200 random eval clips (silence and classical features on one CPU core; XLS-R on the Apple M5 GPU,
`mps`), training time and scoring latency from the recorded runs (`results/rung*.csv`), and model
size (serialized bytes, trainable and frozen parameters). Output: `results/cost_table.csv`.

**Final model** = rung 2 (frozen XLS-R-300M layer 3 + logistic regression), the best model on ASVspoof
eval. Saved to `model/` with its metadata (layer, dev EER threshold, results, sha256); scored with
`python -m deepvoice.predict`. See `model/MODEL_CARD.md`.

In [ ]:
import hashlib
import io
import json
import os
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
from deepvoice import predict, xlsr
from deepvoice.features import clip_features, load_audio, load_xy
from deepvoice.metrics import eer
from deepvoice.models import build
from deepvoice.silence import silence_features

p = pd.read_parquet("data/protocols.parquet")
sample = list(p[p.split == "eval"].sample(200, random_state=0).path)

## Feature extraction time per clip

In [ ]:
def per_clip_ms(fn):
    fn(sample[0])  # warm-up
    t0 = time.perf_counter()
    for path in sample:
        fn(path)
    return (time.perf_counter() - t0) / len(sample) * 1000

ext = {0: per_clip_ms(lambda q: silence_features(load_audio(q))),
       1: per_clip_ms(clip_features)}  # computes MFCC + LFCC + spectral: an upper bound for LFCC alone
xm = xlsr.load_model()
ext[2] = xlsr.time_it(sample, xm)["s_per_clip"] * 1000
frozen_params = sum(t.numel() for t in xm.parameters())
{k: round(v, 1) for k, v in ext.items()}

## The three final models: refit (deterministic, seed 0), size, recorded train and scoring time

In [ ]:
gap = pd.read_csv("results/e3_gap.csv")
hgb_params = json.loads(gap[(gap.model == "hgb") & (gap["selected by"] == "dev")].params.iloc[0])
SIL = ["lead_sil", "trail_sil", "sil_ratio", "duration"]
sil = {s: load_xy(s, groups=("silence",)) for s in ["train", "dev"]}
lf = {s: load_xy(s, groups=("lfcc",), n_coef=40) for s in ["train", "dev"]}
LF = list(lf["train"].columns[4:])
K = xlsr.LAYERS.index(3)
xl = {s: np.asarray(np.load(f"data/features/xlsr300m_{s}.npy", mmap_mode="r")[:, K], dtype=np.float32) for s in ["train", "dev"]}
y = {s: sil[s].label.to_numpy() for s in ["train", "dev"]}

m0 = build("tree", 0, max_depth=3).fit(sil["train"][SIL].to_numpy(), y["train"])
m1 = build("hgb", 0, **hgb_params).fit(lf["train"][LF].to_numpy(), y["train"])
m2 = build("logreg", 0, C=1.0).fit(xl["train"], y["train"])

def nbytes(m):
    b = io.BytesIO(); joblib.dump(m, b); return b.tell()

trainable = {0: int(m0.tree_.node_count),
             1: int(sum(len(pr[0].nodes) for pr in m1._predictors)),
             2: int(m2[-1].coef_.size + m2[-1].intercept_.size)}
CONFIG = {0: ("rung0", "rung0_tree_s0"), 1: ("rung1", "rung1_hgb_lfcc40dd_e3devsel_s0"), 2: ("rung2", "rung2_logreg_xlsr300m_L03_s0")}
rows = []
for rung, (exp, cfg) in CONFIG.items():
    r = pd.read_csv(f"results/{exp}.csv").query("config == @cfg").set_index("split")
    rows.append({
        "rung": rung,
        "model": {0: "depth-3 tree, 4 silence/duration numbers", 1: "gradient boosting, LFCC-40 + deltas (240)",
                  2: "frozen XLS-R-300M layer 3 + logistic regression"}[rung],
        "extract_ms_per_clip": ext[rung], "extract_device": "mps (GPU)" if rung == 2 else "1 CPU core",
        "train_s": r.loc["dev", "train_time"], "score_ms_per_clip": r.loc["eval", "latency"],
        "model_kb": nbytes([m0, m1, m2][rung]) / 1024, "trainable_params_or_nodes": trainable[rung],
        "frozen_params": frozen_params if rung == 2 else 0,
        "eval_eer_pct": 100 * r.loc["eval", "eer"], "itw_eer_pct": 100 * r.loc["itw", "eer"]})
cost = pd.DataFrame(rows)
cost.to_csv("results/cost_table.csv", index=False, float_format="%.4g")
cost.round(3)

## Save the final model (rung 2) and check the scorer reproduces the pipeline

In [ ]:
s_dev = m2.predict_proba(xl["dev"])[:, 1]
dev_eer, thr = eer(y["dev"], s_dev)
Path("model").mkdir(exist_ok=True)
joblib.dump(m2, Path("model") / predict.MODEL_FILE)
sha = hashlib.sha256((Path("model") / predict.MODEL_FILE).read_bytes()).hexdigest()
r2 = pd.read_csv("results/rung2.csv").query("config == 'rung2_logreg_xlsr300m_L03_s0'").set_index("split")
meta = {"name": "DeepVoice rung 2", "xlsr_model": xlsr.MODEL, "layer": 3, "pooling": "mean over time",
        "classifier": "StandardScaler + LogisticRegression(C=1, class_weight=balanced)",
        "train_data": "ASVspoof 2019 LA train (25,380 clips, 16 kHz mono)", "seed": 0,
        "threshold": thr, "threshold_rule": "EER threshold on ASVspoof 2019 LA dev",
        "eer": {s: round(float(r2.loc[s, "eer"]), 6) for s in ["dev", "eval", "itw"]},
        "eer_ci95": {s: [round(float(r2.loc[s, "ci_low"]), 6), round(float(r2.loc[s, "ci_high"]), 6)] for s in ["eval", "itw"]},
        "sha256": sha}
(Path("model") / predict.MODEL_FILE.replace(".joblib", ".json")).write_text(json.dumps(meta, indent=2) + "\n")

# the CLI path (load file -> XLS-R -> layer 3 -> classifier) must give the pipeline's scores
check = p[p.split == "dev"].reset_index(drop=True).sample(8, random_state=1)
clf, meta_loaded = predict.load()
got = predict.score(check.path.tolist(), clf, meta_loaded, xm)
want = m2.predict_proba(xl["dev"][check.index])[:, 1]
print("max |difference| CLI vs pipeline:", float(np.max(np.abs(got - want))))
assert np.allclose(got, want, atol=0.02)
meta